# 03. BFS with a Queue

**Breadth-First Search** explores a structure **level by level**: all nodes at distance 1, then distance 2, and so on. A **queue** gives exactly this order. In an **unweighted** graph BFS finds the **shortest path**.

**Learning order:** REAL LIFE → TEMPLATE → TREE LEVEL ORDER → GRAPH DISTANCES → GRID SHORTEST PATH → MULTI-SOURCE BFS → INTERVIEW THINKING

## 1. Real life

A rumor spreads: you tell your friends (level 1), they tell their friends (level 2)... BFS tells you **how many rounds** until a person hears it. Another example: "minimum number of moves to reach a place" on a map where each move costs 1.

**Why shortest?** BFS reaches every node in the fewest steps first, because it finishes all nodes at distance `d` before any node at distance `d + 1`.

## 2. Template

```python
q = deque([start]); visited = {start}
while q:
    for _ in range(len(q)):          # process one whole level
        node = q.popleft()
        for nb in neighbours(node):
            if nb not in visited:
                visited.add(nb)      # mark when you ENQUEUE, not when you pop
                q.append(nb)
    level += 1
```
Mark visited **when adding to the queue**, otherwise the same node can be added many times.

In [1]:
from collections import deque

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def tree_from_list(vals):             # level order, None = missing child
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    q, i = deque([root]), 1
    while q and i < len(vals):
        node = q.popleft()
        if i < len(vals) and vals[i] is not None:
            node.left = TreeNode(vals[i]); q.append(node.left)
        i += 1
        if i < len(vals) and vals[i] is not None:
            node.right = TreeNode(vals[i]); q.append(node.right)
        i += 1
    return root

def level_order(root):
    if not root:
        return []
    q, res = deque([root]), []
    while q:
        level = []
        for _ in range(len(q)):                 # fixed size of the current level
            node = q.popleft()
            level.append(node.val)
            if node.left:  q.append(node.left)
            if node.right: q.append(node.right)
        res.append(level)
    return res

print(level_order(tree_from_list([3, 9, 20, None, None, 15, 7])))   # [[3], [9, 20], [15, 7]]

[[3], [9, 20], [15, 7]]


## 3. Shortest distance in a graph

Graph as an **adjacency list** (dictionary: node → neighbours). `dist[node]` = number of edges from the start.

In [2]:
def bfs_distances(graph, start):
    dist = {start: 0}
    q = deque([start])
    while q:
        node = q.popleft()
        for nb in graph[node]:
            if nb not in dist:
                dist[nb] = dist[node] + 1
                q.append(nb)
    return dist

graph = {"A": ["B", "C"], "B": ["D"], "C": ["D", "E"], "D": ["F"], "E": ["F"], "F": []}
print(bfs_distances(graph, "A"))   # {'A': 0, 'B': 1, 'C': 1, 'D': 2, 'E': 2, 'F': 3}

{'A': 0, 'B': 1, 'C': 1, 'D': 2, 'E': 2, 'F': 3}


## 4. Shortest path in a grid (maze)

Cells are `0` (free) or `1` (wall). Move up/down/left/right. Each cell is a node with up to 4 neighbours. Use `(row, col)` tuples and a `directions` list.

In [3]:
def shortest_path_grid(grid):
    rows, cols = len(grid), len(grid[0])
    if grid[0][0] == 1 or grid[rows - 1][cols - 1] == 1:
        return -1
    q = deque([(0, 0, 0)])                       # row, col, steps
    seen = {(0, 0)}
    while q:
        r, c, steps = q.popleft()
        if (r, c) == (rows - 1, cols - 1):
            return steps
        for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nr, nc = r + dr, c + dc
            if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 0 and (nr, nc) not in seen:
                seen.add((nr, nc))
                q.append((nr, nc, steps + 1))
    return -1

maze = [[0, 0, 1], [1, 0, 1], [1, 0, 0]]
print(shortest_path_grid(maze))   # 4

4


## 5. Multi-source BFS: Rotting Oranges

Several rotten oranges spread rot **at the same time**. Put **all sources** in the queue at the start (level 0). Each level = one minute. At the end, if a fresh orange remains, return `-1`.

Real life: fire spreading from many places; wifi signals from many routers.

In [4]:
def oranges_rotting(grid):
    rows, cols = len(grid), len(grid[0])
    q, fresh = deque(), 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == 2:
                q.append((r, c))
            elif grid[r][c] == 1:
                fresh += 1
    minutes = 0
    while q and fresh:
        for _ in range(len(q)):
            r, c = q.popleft()
            for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                nr, nc = r + dr, c + dc
                if 0 <= nr < rows and 0 <= nc < cols and grid[nr][nc] == 1:
                    grid[nr][nc] = 2
                    fresh -= 1
                    q.append((nr, nc))
        minutes += 1
    return -1 if fresh else minutes

print(oranges_rotting([[2, 1, 1], [1, 1, 0], [0, 1, 1]]))   # 4
print(oranges_rotting([[2, 1, 1], [0, 1, 1], [1, 0, 1]]))   # -1
print(oranges_rotting([[0, 2]]))                            # 0

4
-1
0


## 6. Interview thinking

**Signals:** "minimum number of steps/moves/hops", "level order", "nearest", "spread from several sources", unweighted shortest path.

**Say:** "Every move costs 1, so BFS reaches each cell in the minimum steps. I mark cells visited when I enqueue them. Time O(V + E), space O(V)."

**Common mistakes:** marking visited on pop (duplicates in the queue), `list.pop(0)`, forgetting bounds checks, not processing level by level when the answer is a level count, using BFS for **weighted** graphs (use Dijkstra).

## Quick Revision

- BFS = queue, level by level. Shortest path in unweighted graphs.
- Mark visited when enqueuing. Level loop: `for _ in range(len(q))`.
- Grid: 4 directions, bounds check, `seen` set.
- Multi-source: put all sources in the queue first.
- O(V + E). Weighted → Dijkstra.